# 实战练习：使用 Unsloth 加速 GRPO 训练

本节将使用 **Unsloth** 加速 GRPO 训练，在 Google Colab 免费 **T4 GPU** 上也能运行！

Unsloth 是一个专门为 LLM 微调加速设计的库：
- 训练速度提升 2x~5x
- 显存占用降低 60%
- 无缝集成 TRL，兼容 GRPOTrainer
- 支持 4-bit 量化推理

**本练习使用的资源：**
- 模型：`google/gemma-3-1b-it`（1B 参数 Instruct 模型）
- 数据集：`openai/gsm8k`（小学数学题）
- 任务：训练模型先展示推理过程，再给出答案
- 硬件：Google Colab 免费 T4 GPU

> **TIP**: 推荐在 Google Colab 上跟随本练习操作，体验完整的训练过程。

## 环境安装

In [ ]:
# 安装 Unsloth 和 vLLM
# unsloth: 加速微调库
# vllm: 高效推理引擎（用于 GRPO 的快速生成阶段）
!pip install unsloth vllm
!pip install --upgrade pillow  # 确保 pillow 版本兼容

## 加载模型（Unsloth 方式）

Unsloth 提供了 `FastLanguageModel` 类，整合了模型加载、量化和 LoRA 配置：

In [ ]:
from unsloth import FastLanguageModel
import torch

# 训练配置
max_seq_length = 1024  # 支持更长的推理链（reasoning traces）
lora_rank = 32         # LoRA 秩，越大模型容量越强但越慢

# 使用 Unsloth 的 FastLanguageModel 加载 Gemma 3 1B Instruct 模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-3-1b-it",   # 使用 Google Gemma 3 1B 指令模型
    max_seq_length=max_seq_length,
    load_in_4bit=True,                    # 4-bit 量化（显著节省显存）
    fast_inference=True,                  # 启用 vLLM 快速推理（加速 GRPO 生成阶段）
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,           # GPU 显存使用率（如 OOM 可适当降低）
)

# 应用 LoRA 配置
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        # 对注意力层的所有投影矩阵应用 LoRA
        "q_proj",   # Query 投影
        "k_proj",   # Key 投影
        "v_proj",   # Value 投影
        "o_proj",   # 输出投影
        # 对 FFN 层应用 LoRA
        "gate_proj",
        "up_proj",
        "down_proj",
        # 注：如果显存不足，可移除 q/k/v/o 部分
    ],
    lora_alpha=lora_rank,                 # 通常设为与 r 相同的值
    use_gradient_checkpointing="unsloth", # 使用 Unsloth 优化的梯度检查点（支持更长上下文）
    random_state=3407,                    # 固定随机种子，保证可复现
)

print("模型加载完成！")
print(f"使用设备：{next(model.parameters()).device}")

## 数据准备

使用 GSM8K 数据集（小学数学题）。我们需要格式化数据，让模型学会在给出答案前先展示推理过程。

### 目标格式

训练后，模型应该以以下格式回答数学题：

```
<thinking>
先算 2×6=12，再加 2，得到 14。
</thinking>

<answer>
14
</answer>
```

In [ ]:
# 定义系统提示词：指导模型使用 <thinking>/<answer> 格式回答
SYSTEM_PROMPT = """
Respond in the following format:

<thinking>
...
</thinking>

<answer>
...
</answer>

"""

# 目标输出格式模板（仅用于参考，不直接用于训练）
XML_COT_FORMAT = """\
<thinking>
{reasoning}
</thinking>

<answer>
{answer}
</answer>
"""

print("系统提示词：")
print(SYSTEM_PROMPT)

In [ ]:
import re
from datasets import load_dataset, Dataset

def extract_xml_answer(text: str) -> str:
    """
    从 <answer>...</answer> 标签中提取答案
    用于从模型输出中解析最终答案
    """
    answer = text.split("<answer>")[-1]   # 取最后一个 <answer> 之后的内容
    answer = answer.split("</answer>")[0]  # 取 </answer> 之前的内容
    return answer.strip()


def extract_hash_answer(text: str) -> str | None:
    """
    从 GSM8K 格式的答案中提取数字答案
    GSM8K 答案格式：「...计算步骤... #### 答案数字」
    """
    if "####" not in text:
        return None  # 格式不匹配，返回 None
    return text.split("####")[1].strip()  # 提取 #### 后的数字


def get_gsm8k_questions(split="train") -> Dataset:
    """
    加载并预处理 GSM8K 数学数据集
    
    将原始数据转换为包含以下字段的格式：
    - prompt: 带系统提示的消息列表（供模型生成回答）
    - answer: 正确的数字答案（供奖励函数验证）
    """
    # 加载 GSM8K 数据集（main 配置包含 CoT 格式的答案）
    data = load_dataset("openai/gsm8k", "main")[split]
    
    # 转换数据格式
    data = data.map(
        lambda x: {
            # 构造消息格式（包含系统提示和用户问题）
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},  # 系统提示
                {"role": "user", "content": x["question"]},    # 用户问题
            ],
            # 提取标准答案（只保留数字部分）
            "answer": extract_hash_answer(x["answer"]),
        }
    )
    return data


# 加载数据集
dataset = get_gsm8k_questions()

print(f"数据集大小：{len(dataset)}")
print()
print("第一个样本：")
print(f"问题：{dataset[0]['prompt'][1]['content']}")
print(f"正确答案：{dataset[0]['answer']}")

## 定义奖励函数

本练习使用**多个奖励函数的组合**，从不同角度引导模型：

| 奖励函数 | 目的 | 最高分 |
|----------|------|--------|
| `correctness_reward_func` | 答案正确性验证 | 2.0 |
| `int_reward_func` | 鼓励给出数字答案 | 0.5 |
| `strict_format_reward_func` | 严格格式检查 | 0.5 |
| `soft_format_reward_func` | 宽松格式检查 | 0.5 |
| `xmlcount_reward_func` | XML 标签完整性 | 0.5 |

总分最高 4.0，正确性奖励权重最大。

In [ ]:
import re

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    正确性奖励函数：最重要的奖励，验证答案是否正确
    
    答案正确：+2.0（权重最高）
    答案错误：+0.0
    """
    # 提取每个 completion 的文本内容
    responses = [completion[0]["content"] for completion in completions]
    q = prompts[0][-1]["content"]  # 获取问题文本（用于调试输出）
    
    # 从每个回答中提取 <answer>...</answer> 内的内容
    extracted_responses = [extract_xml_answer(r) for r in responses]
    
    # 打印调试信息（可选，方便观察训练过程）
    print(
        "-" * 20,
        f"问题：\n{q}",
        f"\n正确答案：\n{answer[0]}",
        f"\n模型回答：\n{responses[0]}",
        f"\n提取到的答案：\n{extracted_responses[0]}",
    )
    
    # 对比每个提取答案与正确答案，给予奖励
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]


def int_reward_func(completions, **kwargs) -> list[float]:
    """
    整数奖励函数：鼓励模型给出数字（而非文字）答案
    
    答案是纯数字：+0.5
    答案包含非数字字符：+0.0
    """
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    
    # isdigit() 检查字符串是否全为数字字符
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]


def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """
    严格格式奖励函数：检查是否完全符合期望格式
    格式要求：\n<thinking>\n...\n</thinking>\n\n<answer>\n...\n</answer>\n
    
    完全符合格式：+0.5
    不符合：+0.0
    """
    # 严格的正则表达式（要求精确的换行符位置）
    pattern = r"^\n<thinking>\n.*?\n</thinking>\n\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]


def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """
    宽松格式奖励函数：只要包含基本的 XML 标签结构即可
    格式要求：包含 <thinking>...</thinking> 和 <answer>...</answer>
    
    包含基本标签结构：+0.5
    缺少标签：+0.0
    """
    # 宽松的正则表达式（允许标签间有任意空白）
    pattern = r"<thinking>.*?</thinking>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]


def count_xml(text) -> float:
    """
    XML 标签计数函数：计算各 XML 标签的完整性
    
    每个正确的标签 +0.125 分（共 4 个标签，满分 0.5）
    额外惩罚：</answer> 之后若有多余内容，每个字符 -0.001 分
    （防止模型在答案后添加无关内容）
    """
    count = 0.0
    
    # 检查开始标签：<thinking>\n 出现恰好一次
    if text.count("<thinking>\n") == 1:
        count += 0.125
    
    # 检查结束标签：\n</thinking>\n 出现恰好一次
    if text.count("\n</thinking>\n") == 1:
        count += 0.125
    
    # 检查答案开始标签：\n<answer>\n 出现恰好一次
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        # 惩罚 </answer> 之后的多余内容
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001
    
    # 检查答案结束标签：\n</answer> 出现恰好一次
    if text.count("\n</answer>") == 1:
        count += 0.125
        # 惩罚 </answer> 之后的多余内容（不计最后的换行符）
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001
    
    return count


def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    """
    XML 标签完整性奖励函数：用 count_xml 评估 XML 格式完整性
    满分 0.5，鼓励精确的格式使用
    """
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]


print("奖励函数定义完成！")
print()
print("各奖励函数说明：")
print(f"  correctness_reward_func: 正确答案 → +2.0（最重要）")
print(f"  int_reward_func:         纯数字答案 → +0.5")
print(f"  strict_format_reward_func: 严格格式 → +0.5")
print(f"  soft_format_reward_func: 宽松格式 → +0.5")
print(f"  xmlcount_reward_func:    XML 完整性 → 最高 +0.5")
print(f"  总计最高得分：+4.0")

## 配置并启动 GRPO 训练

In [ ]:
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = 256  # prompt 最大长度（比较短，节省显存）

# 配置 GRPO 训练参数
training_args = GRPOConfig(
    # ---- 优化器参数 ----
    learning_rate=5e-6,                                   # 学习率（较小，避免过拟合）
    adam_beta1=0.9,                                        # Adam 动量参数
    adam_beta2=0.99,                                       # Adam 自适应学习率参数
    weight_decay=0.1,                                      # 权重衰减（L2 正则化）
    warmup_ratio=0.1,                                      # 学习率预热比例（前 10% 步数线性增大）
    lr_scheduler_type="cosine",                            # 余弦学习率调度
    optim="paged_adamw_8bit",                              # 分页 8-bit AdamW（节省显存）
    
    # ---- 批次参数 ----
    per_device_train_batch_size=1,                         # T4 显存有限，设为 1
    gradient_accumulation_steps=1,                         # 可增大到 4 获得更平滑的训练曲线
    
    # ---- GRPO 核心参数 ----
    num_generations=6,                                     # 每题生成 6 个候选（显存不足可调低）
    max_prompt_length=max_prompt_length,
    max_completion_length=max_seq_length - max_prompt_length,  # 生成长度 = 总长度 - prompt 长度
    
    # ---- 训练控制 ----
    max_steps=250,                                         # 总训练步数（非完整 epoch，快速验证）
    save_steps=250,                                        # 每 250 步保存一次检查点
    max_grad_norm=0.1,                                     # 梯度裁剪（防止梯度爆炸）
    
    # ---- 日志 ----
    logging_steps=1,                                       # 每步记录日志
    report_to="wandb",                                     # 启用 W&B 追踪
    run_name="gemma-3-1b-it-grpo-gsm8k",                  # 实验名称（W&B 中显示）
    output_dir="outputs",
)

hub_model_name = 'goosmanlei/gemma-3-1b-it-grpo-gsm8k'
# 初始化 GRPOTrainer，传入所有奖励函数
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        xmlcount_reward_func,          # XML 完整性（0.5）
        soft_format_reward_func,        # 宽松格式（0.5）
        strict_format_reward_func,      # 严格格式（0.5）
        int_reward_func,               # 整数答案（0.5）
        correctness_reward_func,        # 正确性（2.0，最重要）
    ],
    args=training_args,
    train_dataset=dataset,
    push_to_hub=True,
    hub_name=hub_model_name,
)

print("GRPOTrainer 初始化完成！")

In [ ]:
# 开始训练！
# 在 T4 GPU 上，250 步约需 15~30 分钟
# 注意：前 150~200 步可能看不到明显的奖励提升，请耐心等待！
trainer.train()

> **WARNING**: 训练初期（前 150~200 步）可能看不到奖励上升。这是正常的——模型需要时间学习格式和推理模式。请保持耐心！

## 测试训练好的模型

In [ ]:
# 保存 LoRA 权重
model.save_lora("grpo_saved_lora")
print("LoRA 权重已保存到 grpo_saved_lora/")

In [ ]:
from vllm import SamplingParams

# 准备测试问题（注意：这是一个开放性问题，非 GSM8K 格式）
test_question = "Calculate pi."

# 应用 chat template 格式化输入
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_question},
    ],
    tokenize=False,
    add_generation_prompt=True,  # 添加生成提示（触发模型生成回答）
)

# 推理参数
sampling_params = SamplingParams(
    temperature=0.8,   # 适中的温度，兼顾多样性和准确性
    top_p=0.95,        # 只保留累积概率 95% 以内的 token
    max_tokens=1024,   # 最大生成长度（足够展示完整推理链）
)

# 使用 vLLM 快速推理
output = (
    model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=model.load_lora("grpo_saved_lora"),  # 加载已训练的 LoRA 权重
    )[0]
    .outputs[0]
    .text
)

print(f"问题：{test_question}")
print()
print("模型回答：")
print(output)
print()
print("观察：训练后的模型是否使用了 <thinking>...</thinking><answer>...</answer> 格式？")

## 保存与发布模型

Unsloth 提供多种保存格式，满足不同使用场景：

In [ ]:
# --------------------------------------------------------
# 方式 1：保存为 16-bit 精度（标准 HuggingFace 格式）
# 适合：需要完整精度进行推理或进一步微调
# --------------------------------------------------------
model.save_pretrained_merged("model", tokenizer, save_method="merged_16bit")
print("已保存 16-bit 精度模型到 ./model/")

In [ ]:
# --------------------------------------------------------
# 方式 2：推送到 HuggingFace Hub（16-bit 格式）
# 适合：分享模型供他人使用
# --------------------------------------------------------
# model.push_to_hub_merged(
#     "your-username/model-name",       # 替换为你的用户名和模型名
#     tokenizer,
#     save_method="merged_16bit",
#     token="your-token"                # 替换为你的 HuggingFace token
# )

# --------------------------------------------------------
# 方式 3：推送为 GGUF 格式（用于本地推理）
# 适合：在本地用 llama.cpp、Jan、Open WebUI 等工具运行
# --------------------------------------------------------
# model.push_to_hub_gguf(
#     "your-username/model-name",
#     tokenizer,
#     # 可同时导出多种量化格式（平衡质量和文件大小）
#     quantization_method=["q4_k_m", "q8_0", "q5_k_m"],
#     token="your-token"
# )

print("保存选项：")
print("  merged_16bit：标准格式，适合进一步微调或高精度推理")
print("  GGUF q4_k_m：4-bit 量化，文件最小（约原始大小 25%）")
print("  GGUF q8_0：8-bit 量化，平衡质量和大小")
print("  GGUF q5_k_m：5-bit 量化，折中选项")

## 本节小结

恭喜完成本章的最后一个练习！

### 你学到了什么

1. **Unsloth 加速**：4-bit 量化 + vLLM 推理，在免费 T4 GPU 上运行 1B 模型

2. **数据准备**：如何格式化 GSM8K 数学数据集，引导模型学习推理格式

3. **多维度奖励设计**：组合 5 个奖励函数（正确性 + 格式 + 结构），从不同角度引导训练

4. **模型保存**：支持 16-bit 和 GGUF 等多种格式，适应不同部署场景

### 奖励函数组合策略总结

| 阶段 | 奖励函数 | 权重 |
|------|----------|------|
| 格式学习（初期）| xmlcount + soft_format + strict_format | 各 0.5 |
| 数值校验（中期）| int_reward | 0.5 |
| 正确性优化（全程）| correctness | 2.0 |

### 参考资料

- [Unsloth 官方文档](https://docs.unsloth.ai/)
- [Unsloth Discord 社区](https://discord.gg/unsloth)
- [Unsloth GitHub 仓库](https://github.com/unslothai/unsloth)
- [TRL GRPO 文档](https://huggingface.co/docs/trl/main/en/grpo_trainer)

---

## Chapter 12 总结

你已完成 Chapter 12 的全部学习内容！回顾本章核心知识：

1. **强化学习基础**：Agent、Environment、Action、Reward、Policy 五大要素
2. **RLHF 原理**：通过人类反馈引导 LLM 生成更符合期望的输出
3. **GRPO 算法**：组内相对比较 + 无 Critic 网络 + 灵活奖励函数
4. **DeepSeek R1**：4 阶段训练 + 「顿悟时刻」现象 + 强大推理能力
5. **实践技能**：用 TRL 和 Unsloth 在有限资源上完成 GRPO 微调